In [1]:
import os
from datetime import datetime
import sys
from pathlib import Path
import shutil
import zipfile
import numpy as np
import pandas as pd
import tqdm
from tqdm.auto import tqdm
from thefuzz import process
import warnings as wr
wr.filterwarnings("ignore")

In [2]:
def move_dir_to_dir(dir_to_move, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    dir_name = dir_to_move.name
    dest_path = dest_dir / dir_name
    os.makedirs(dest_path, exist_ok=True)
    for item in dir_to_move.iterdir():
        item_name = item.name
        item_dest_path = dest_path / item_name
        shutil.move(item, item_dest_path)
    if not any(dir_to_move.iterdir()):
        shutil.rmtree(dir_to_move)

In [3]:
def rc123_basic(rc1, rc2, rc3, unique_list):
    yr, state, season, st, distt, stra, vill, uid = unique_list
    
    #setting basics values for rc1
    rc1.loc[0, 'REJ'] = 0
    rc1.loc[0, 'RC'] = 1
    rc1.loc[0, 'ST'] = st
    rc1.loc[0, 'YR'] = yr
    rc1.loc[0, 'SESON'] = season
    rc1.loc[0, 'STAT'] = state
    rc1.loc[0, 'DIST'] = distt
    rc1.loc[0, 'STRA'] = stra
    rc1.loc[0, 'VILL'] = vill
    rc1.loc[0, 'ID'] = uid
    
    #setting basics values for rc2
    rc2.loc[0, 'RC'] = 2
    rc2.loc[0, 'ST'] = st
    rc2.loc[0, 'YR'] = yr
    rc2.loc[0, 'SESON'] = season
    rc2.loc[0, 'STAT'] = state
    rc2.loc[0, 'DIST'] = distt
    rc2.loc[0, 'STRA'] = stra
    rc2.loc[0, 'VILL'] = vill
    rc2.loc[0, 'ID'] = uid
    
    #setting basics values for rc3
    rc3.loc[0, 'RC'] = 3
    rc3.loc[0, 'ST'] = st
    rc3.loc[0, 'YR'] = yr
    rc3.loc[0, 'SESON'] = season
    rc3.loc[0, 'STAT'] = state
    rc3.loc[0, 'DIST'] = distt
    rc3.loc[0, 'STRA'] = stra
    rc3.loc[0, 'VILL'] = vill
    rc3.loc[0, 'ID'] = uid

    return rc1, rc2, rc3

In [4]:
def comment_box_file(rc1, df):
    reason_map = {
        'system of girdawari does not exist' : 0,
        'girdawari not done for previous year/current year' : 1,
        'khasra register/other records of statements (specify) not available' : 2,
        'girdawari completed but jinswar/trs statement not prepared' : 3,
        'reason for non availability of information not known' : 4,
        'sample village is non trs village' : 5,
        'not applicable for kerala, orissa and hilly districts of uttar pradesh' : 6,
        'aggregation figures not available at village level for kerala and orissa' : 7,
        'other reasons (specify)' : 8,
        '' : 9
    }
    
    rc61_reason = str(df.iloc[0, 1]).lower()
    if not rc61_reason or rc61_reason.strip() == "":
        rc1.loc[0, 'RC61'] = reason_map['']
    else:
        best_match_key, score = process.extractOne(rc61_reason.strip(), reason_map.keys())
        if score >= 60:
            rc61_code = reason_map[best_match_key]
        else:
            rc61_code = 9
        rc1.loc[0, 'RC61'] = rc61_code
    
    rc62_reason = str(df.iloc[0, 1]).lower()
    if not rc62_reason or rc62_reason.strip() == "":
        rc1.loc[0, 'RC62'] = reason_map['']
    else:
        best_match_key, score = process.extractOne(rc62_reason.strip(), reason_map.keys())
        if score >= 60:
            rc62_code = reason_map[best_match_key]
        else:
            rc62_code = 9
        rc1.loc[0, 'RC62'] = rc62_code
    
    rc63_reason = str(df.iloc[0, 1]).lower()
    if not rc63_reason or rc63_reason.strip() == "":
        rc1.loc[0, 'RC63'] = reason_map['']
    else:
        best_match_key, score = process.extractOne(rc63_reason.strip(), reason_map.keys())
        if score >= 60:
            rc63_code = reason_map[best_match_key]
        else:
            rc63_code = 9
        rc1.loc[0, 'RC63'] = rc63_code
    
    return rc1

In [5]:
def basic_file_block_11(rc1, df, unique_list):
    yr = unique_list[0]
    
    mac = df.loc[0, 'mawp']
    rc1.loc[0, 'MAC'] = mac
    if mac == 1:
        rc1.loc[0, 'MUC'] = df.loc[0, 'usable']
    
    cadc = df.loc[0, 'cs']
    rc1.loc[0, 'CADC'] = cadc
    if cadc in (3,4,9):
        rc1.loc[0, 'TLC'] = 0
    elif cadc in (1,2):
        map_upd_year = df.loc[0, 'muy']
        if map_upd_year == None:
            rc1.loc[0, 'TLC'] = 9
        else:
            # calculating number of years from map update date
            if len(str(map_upd_year))>4:
                map_upd_year = int(str(map_upd_year)[:4]) + 1
            
            yr_gap = yr - map_upd_year
            if yr_gap <= 1:
                rc1.loc[0, 'TLC'] = 1
            elif yr_gap <= 5:
                rc1.loc[0, 'TLC'] = 2
            elif yr_gap <= 10:
                rc1.loc[0, 'TLC'] = 3
            elif yr_gap <= 20:
                rc1.loc[0, 'TLC'] = 4
            elif yr_gap > 20:
                rc1.loc[0, 'TLC'] = 5
    
    #############################################
    #### SGC, DDG, ADG, GCC
    #############################################
    df['gdoc'] = pd.to_datetime(df['gdoc'], format='%d/%m/%y', errors='coerce')
    df['actualdate'] = pd.to_datetime(df['actualdate'], format='%d/%m/%y', errors='coerce')
    ddg_str = str(df.loc[0, 'gdoc'])
    sgc = df.loc[0, 'gsoc']
    adg_str = str(df.loc[0, 'actualdate'])
    
    ddg_dt = df.loc[0, 'gdoc']
    adg_dt = df.loc[0, 'actualdate']
    
    rcfgnc = df['grfnc']    #reason code for girdawari not completed 10(d)
    
    # calculation of GCC
    gcc = 999
    if sgc == 1:
        if adg_str == '':
            gcc = 3
        elif adg_dt <= ddg_dt:
            gcc = 1
        elif adg_dt > ddg_dt:
            gcc = 2
    elif sgc in (2,3):
        if adg_str == '':
            if rcfgnc == 1:
                gcc = 4
            elif rcfgnc == 2:
                gcc = 5
            elif rcfgnc == 9:
                gcc = 6
            elif rcfgnc == None:
                gcc = 7
    elif sgc == None and adg_str == '':
        gcc = 9
    else:
        gcc = 8
    
    if sgc == 1:
        rc1.loc[0, 'EPC'] = 1
    else:
        rc1.loc[0, 'EPC'] = 3
    
    rc1.loc[0, 'SGC'] = sgc
    rc1.loc[0, 'DDG'] = f"{ddg_str[:2]}{ddg_str[3:5]}"
    rc1.loc[0, 'GCC'] = gcc
    rc1.loc[0, 'LFOG'] = df.loc[0, 'lfogu']
    rc1.loc[0, 'ROG'] = df.loc[0, 'rogk']
    
    #############################################
    #### DDTRS, ADTRS, TRSSF, TRSSC
    #############################################
    df['ddfst'] = pd.to_datetime(df['ddfst'], format='%d/%m/%y', errors='coerce')
    df['iyados'] = pd.to_datetime(df['iyados'], format='%d/%m/%y', errors='coerce')
    
    ddtrs_str = str(df.loc[0, 'ddfst'])
    trs = df.loc[0, 'tssta']
    adtrs_str = str(df.loc[0, 'iyados'])
    
    ddtrs_dt = df.loc[0, 'ddfst']
    adtrs_dt = df.loc[0, 'iyados']

    trssc = 999
    if ddtrs_str != '' and adtrs_str != '':
        if adtrs_dt <= ddtrs_dt:
            if adtrs_dt < adg_dt or (adg_str == '' and sgc in (2,3) ):
                trssc = 0
            elif adtrs_dt >= adg_dt:
                trssc = 1
            elif sgc == 1 and adg_str == '':
                trssc = 2
        elif adtrs_dt > ddtrs_dt:
            if (adg_str == '' or adtrs_dt < adg_dt) and sgc in (2,3):
                trssc = 3
            elif adtrs_dt >= adg_dt:
                trssc = 4
            elif sgc == 1 and adg_str == '':
                trssc = 5
    elif adtrs_str == '':
        if trs == 1:
            trssc = 6
    elif trs == 0:
        if sgc == 1:
            trssc = 7
        elif sgc in (2,3):
            trssc = 8
    elif (trs in ('', None) and adtrs_str == ''):
        trssc = 9
    elif sgc == 1 and trs == 0 and adtrs_dt < ddtrs_dt:
        trssc = 10
    else:
        trssc = 9     #if no condition matches
    
    rc1.loc[0, 'TRSSC'] = trssc
    rc1.loc[0, 'DDTRS'] = f"{ddtrs_str[:2]}{ddtrs_str[3:5]}"
    rc1.loc[0, 'TRSSF'] = df.loc[0, 'wteswsisf']
    rc1.loc[0, 'DCKC'] = df.loc[0, 'dcot']
    
    return rc1

In [6]:
def basic_file_block_32(rc1, df):
    geoa_hec = df.loc[0, 'ih1']
    geoa_lu = df.loc[0, 'ilu11']
    if geoa_hec is not None:
        rc1.loc[0, 'GEOA'] = geoa_hec
    else:
        rc1.loc[0, 'GEOA'] = geoa_lu
    return rc1

In [7]:
def get_rc1(csv_file, rc1, df, unique_list):
    if csv_file.name == "commentboxfile6.csv":
        rc1 = comment_box_file(rc1, df)
    if csv_file.name == "basicfileblock11.csv":
        rc1 = basic_file_block_11(rc1, df, unique_list)
    if csv_file.name == "basicfileblock32.csv":
        basic_file_block_32(rc1, df)
    
    if csv_file.name == "basicfileblock2.csv":
        rc1.loc[0, 'HSSN'] = df.loc[0, 'HSN']
    if csv_file.name == "basicfileblock33.csv":
        rc1.loc[0, 'TNSSN'] = df.loc[:, 'confirmation'].value_counts().get('yes', 0)
    return rc1

In [8]:
def get_rc2():
    pass

In [9]:
def get_rc3():
    pass

In [10]:
def load_rc123(rc1_path, rc2_path, rc3_path):
    # if RC1.xlsx already exists, load it, otherwise create a new dataframe
    if rc1_path.exists():
        rc1 = pd.read_excel(rc1_path, sheet_name='RC1')
    else:
        rc1 = pd.DataFrame(columns=["RC","ST","YR","SESON","NOVP","NOVCOD","STAT"
                ,"DIST","STRA","VILL","EPC","NOVPC","NOVTRS","CADC"
                ,"TLC","MAC","MUC","DDG","SGC","GCC","LFOG","TCROPA"
                ,"ROG","DDTRS","TRSSC","TRSSF","DCKC","HSSN","GEOA"
                ,"TNSSN","TGEOASN","RC61","RC62","RC63","REJ","ID"])
    
    # if RC2.xlsx already exists, load it, otherwise create a new dataframe
    if rc2_path.exists():
        rc2 = pd.read_excel(rc2_path, sheet_name='RC2')
    else:
        rc2 = pd.DataFrame(columns=["RC","ST","YR","SESON","STAT","DIST","STRA"
                        ,"VILL","EPC","HSSN","TNSSN","CROP","VARC","ARSU"
                        ,"ARSI","ARPU","ARPI","ID"])
    
    # if RC3.xlsx already exists, load it, otherwise create a new dataframe
    if rc3_path.exists():
        rc3 = pd.read_excel(rc3_path, sheet_name='RC3')
    else:
        rc3 = pd.DataFrame(columns=["RC","ST","YR","SESON","STAT","DIST","STRA"
                        ,"VILL","SN","CROP","VARC","IRRC","ERC","ID"])

    return rc1, rc2, rc3

In [11]:
def get_single_rc123(csv_dir, rc1, rc2, rc3, curr_dir, base_dir, base_err_dir, base_processed_dir):
    try:
        #for item in csv_dir.glob("*.csv"):
        for item in csv_dir.iterdir():
            if item.name == "BasicFile.csv":
                df = pd.read_csv(item)
                if len(df) > 0:
                    unique_list = []
                    #if "BasicFile" in item.stem:
                    if df['Year','State Code','Sample','District Code','Stratum No',
                           'Order Of Selection'].isna().sum().sum() == 0:
                        # creating unique id
                        year = str(df.loc[0, 'Year'])
                        yr = int(year[:4]) + 1
                        state = str(df.loc[0, 'State Code'])
                        season = str(df.loc[0, 'Season Code'])
                        st = str(df.loc[0, 'Sample'])
                        distt = str(df.loc[0, 'District Code'])
                        stra = str(df.loc[0, 'Stratum No'])
                        vill = str(df.loc[0, 'Order Of Selection'])
                        
                        uid = f"{yr}_{state}_{season}_{st}_{distt}_{stra}_{vill}"
                        unique_list = [yr, state, season, st, distt, stra, vill, uid]
                        
                        #rc1.loc[0, 'NOVPC'] = df.loc[0, 'Number Of Village Total']
                        #rc1.loc[0, 'NOVTRS'] = df.loc[0, 'Number Of Village TRS']
                        
                        #main method to get rc123 data
                        rc1, rc2, rc3 = rc123_basic(rc1, rc2, rc3, unique_list)
                    else:
                        print("1")
                        move_dir_to_dir(csv_dir, base_err_dir)
                else:
                    print("2")
                    move_dir_to_dir(csv_dir, base_err_dir)
            else:
                print("3")
                move_dir_to_dir(csv_dir, base_err_dir)

            #rc1 = get_rc1(item, rc1, df, unique_list)
            #rc2 = get_rc2(item, rc2, df, unique_list)
            #rc3 = get_rc3(item, rc3, df, unique_list)
        
        #move the csv directory to base_processed_dir
        move_dir_to_dir(csv_dir, base_processed_dir)
    except Exception as e:
        pass

    return rc1, rc2, rc3

In [12]:
def create_complete_rc123(rc1_path, rc2_path, rc3_path, curr_dir, 
                          base_dir, base_err_dir, base_processed_dir):
    # load the RCs if exist previously, else create structure
    rc1, rc2, rc3 = load_rc123(rc1_path, rc2_path, rc3_path)

    num_of_items = len(list(base_dir.iterdir()))
    for item in tqdm(base_dir.iterdir(), total=num_of_items, desc="Creating RC123..."):
        if item.is_dir():
            #csv_files = [f for f in item.glob("*.csv")]
            #for csv_file in csv_files:
            #    if "BasicFile" in csv_file.stem:
            
            rc1_s = rc1.iloc[0:0]
            rc2_s = rc2.iloc[0:0]
            rc3_s = rc3.iloc[0:0]
            rc1_s, rc2_s, rc3_s = get_single_rc123(item, rc1_s, rc2_s, rc3_s, curr_dir, 
                                           base_dir, base_err_dir, base_processed_dir)
            # append the row from rc1_s, rc2_s, rc3_s to rc1, rc2, rc3 respectively
            rc1 = pd.concat([rc1, rc1_s])
            rc2 = pd.concat([rc2, rc2_s])
            rc3 = pd.concat([rc3, rc3_s])

    rc1.drop_duplicates(inplace=True)
    rc2.drop_duplicates(inplace=True)
    rc3.drop_duplicates(inplace=True)
    
    rc1.to_excel(rc1_path, sheet_name='RC1', index=False)
    rc2.to_excel(rc2_path, sheet_name='RC2', index=False)
    rc3.to_excel(rc3_path, sheet_name='RC3', index=False)

In [13]:
def main():
    curr_dir = Path.cwd()
    
    base_dir = curr_dir / "05_base_folder"
    base_err_dir = curr_dir / "06_base_error_folder"
    base_processed_dir = curr_dir / "07_base_processed_folder"
    rc_dir = curr_dir / "08_Generated_RCs"
    rc1_path = rc_dir / "RC1.xlsx"
    rc2_path = rc_dir / "RC2.xlsx"
    rc3_path = rc_dir / "RC3.xlsx"
    
    os.makedirs(rc_dir, exist_ok=True)
    os.makedirs(base_processed_dir, exist_ok=True)
    os.makedirs(base_err_dir, exist_ok=True)

    if base_dir.exists():
        create_complete_rc123(rc1_path, rc2_path, rc3_path, curr_dir, base_dir, base_err_dir, base_processed_dir)

In [14]:
main()

Creating RC123...:   0%|          | 0/17517 [00:00<?, ?it/s]

D:\05 GIT\01_masterRepo\06_Office_Related\01_NSO_work\05_AgriSoft_To_RC123\02_Notebooks\05_base_folder\mc_20260801_202940_803923
1
2
D:\05 GIT\01_masterRepo\06_Office_Related\01_NSO_work\05_AgriSoft_To_RC123\02_Notebooks\05_base_folder\mc_20260801_202940_853588
1
2
D:\05 GIT\01_masterRepo\06_Office_Related\01_NSO_work\05_AgriSoft_To_RC123\02_Notebooks\05_base_folder\mc_20260801_202940_904718
1
2
D:\05 GIT\01_masterRepo\06_Office_Related\01_NSO_work\05_AgriSoft_To_RC123\02_Notebooks\05_base_folder\mc_20260801_202940_941619
1
2
D:\05 GIT\01_masterRepo\06_Office_Related\01_NSO_work\05_AgriSoft_To_RC123\02_Notebooks\05_base_folder\mc_20260801_202940_992626
1
2
D:\05 GIT\01_masterRepo\06_Office_Related\01_NSO_work\05_AgriSoft_To_RC123\02_Notebooks\05_base_folder\mc_20260801_202941_104543
1
2
D:\05 GIT\01_masterRepo\06_Office_Related\01_NSO_work\05_AgriSoft_To_RC123\02_Notebooks\05_base_folder\mc_20260801_202941_153718
1
2
D:\05 GIT\01_masterRepo\06_Office_Related\01_NSO_work\05_AgriSoft_To_